# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeda-ujala-haider/FlyRANK-Machine-Learning-First-Assignment/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load data
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

dataset = load_dataset("FlyRank/internship-warehouse",
                       data_files="fact_content_daily_performance_sample.parquet")
df = dataset['train'].to_pandas()

# Filter (same as Week 4)
df_june = df[df['month'] == '2026-06'].copy()
df_clean = df_june[
    (df_june['gsc_data_available'] == True) &
    (df_june['ga4_data_available'] == True) &
    (df_june['gsc_impressions'] >= 10)
].drop_duplicates()

print(f"✅ Data loaded: {len(df_clean)} rows")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

✅ Data loaded: 439193 rows


In [2]:
print("="*80)
print("SECTION 1: KEY FIELD DISTRIBUTIONS")
print("="*80)

# ============================================================
# SIGNAL 1: Impressions (Search Volume)
# ============================================================

print("\n" + "="*80)
print("SIGNAL 1: IMPRESSIONS (Search Volume)")
print("="*80)

impressions = df_clean['gsc_impressions']

print(f"\n STATISTICS:")
print(f"  Count: {len(impressions):,}")
print(f"  Mean: {impressions.mean():.1f}")
print(f"  Median: {impressions.median():.1f}")
print(f"  Std Dev: {impressions.std():.1f}")
print(f"  Min: {impressions.min():.0f}")
print(f"  Max: {impressions.max():.0f}")

print(f"\n PERCENTILES:")
print(f"  10th: {impressions.quantile(0.10):.0f}")
print(f"  25th: {impressions.quantile(0.25):.0f}")
print(f"  50th (median): {impressions.quantile(0.50):.0f}")
print(f"  75th: {impressions.quantile(0.75):.0f}")
print(f"  90th: {impressions.quantile(0.90):.0f}")
print(f"  99th: {impressions.quantile(0.99):.0f}")

print(f"\n BUCKETING (for rule):")
buckets = pd.cut(impressions, bins=[0, 100, 500, 100000], labels=['LOW', 'MED', 'HIGH'])
print(buckets.value_counts().sort_index())

print(f"\n  HEAVY TAIL:")
top_1pct = impressions.quantile(0.99)
top_1pct_count = len(impressions[impressions > top_1pct])
print(f"  Top 1% articles: {impressions.quantile(0.99):.0f}+ impressions")
print(f"  Count: {top_1pct_count} articles")
print(f"  Dominance: {(impressions[impressions > top_1pct].sum() / impressions.sum() * 100):.1f}% of total volume")

print(f"\n INSIGHT:")
print(f"  Volume distribution is RIGHT-SKEWED")
print(f"  Majority: 100-500 impressions (MED)")
print(f"  Small tail: 500+ impressions (HIGH) but HIGH volume dominates impact")

# ============================================================
# SIGNAL 2: CTR Gap
# ============================================================

print("\n" + "="*80)
print("SIGNAL 2: CTR GAP (Expected - Actual)")
print("="*80)

# Calculate CTR gap
position_ctr_benchmark = {
    1: 0.32, 2: 0.26, 3: 0.20, 4: 0.15, 5: 0.12,
    6: 0.10, 7: 0.08, 8: 0.07, 10: 0.05
}

def get_expected_ctr(position):
    position_int = int(position)
    if position_int <= 1:
        return 0.32
    elif position_int >= 10:
        return 0.05
    else:
        return position_ctr_benchmark.get(position_int, 0.10)

df_clean['ctr_expected'] = df_clean['gsc_avg_position'].apply(get_expected_ctr)
df_clean['ctr_actual'] = df_clean['gsc_clicks'] / (df_clean['gsc_impressions'] + 1)
df_clean['ctr_gap'] = df_clean['ctr_expected'] - df_clean['ctr_actual']

ctr_gap = df_clean['ctr_gap']

print(f"\n STATISTICS:")
print(f"  Mean gap: {ctr_gap.mean():.4f} ({ctr_gap.mean()*100:.2f}%)")
print(f"  Median gap: {ctr_gap.median():.4f} ({ctr_gap.median()*100:.2f}%)")
print(f"  Std Dev: {ctr_gap.std():.4f}")
print(f"  Min: {ctr_gap.min():.4f} (over-performing)")
print(f"  Max: {ctr_gap.max():.4f} (massive gap)")

print(f"\n PERCENTILES:")
print(f"  10th: {ctr_gap.quantile(0.10):.4f}")
print(f"  25th: {ctr_gap.quantile(0.25):.4f}")
print(f"  50th: {ctr_gap.quantile(0.50):.4f}")
print(f"  75th: {ctr_gap.quantile(0.75):.4f}")
print(f"  90th: {ctr_gap.quantile(0.90):.4f}")

print(f"\n BUCKETING (for rule):")
gap_buckets = pd.cut(ctr_gap, bins=[-1, 0.04, 0.07, 1], labels=['LOW', 'MED', 'HIGH'])
print(gap_buckets.value_counts().sort_index())

print(f"\n  NEGATIVE GAPS (over-performing):")
negative_gaps = len(ctr_gap[ctr_gap < 0])
print(f"  Articles with gap < 0: {negative_gaps} ({negative_gaps/len(ctr_gap)*100:.1f}%)")
print(f"  These articles rank BETTER than expected")
print(f"  = Already optimized, no refresh needed")

print(f"\n INSIGHT:")
print(f"  Gap distribution shows MOST articles underperform")
print(f"  But {negative_gaps/len(ctr_gap)*100:.1f}% already optimized")
print(f"  Signal is valid but needs threshold tuning")

# ============================================================
# SIGNAL 3: Position (Ranking)
# ============================================================

print("\n" + "="*80)
print("SIGNAL 3: POSITION (Ranking)")
print("="*80)

position = df_clean['gsc_avg_position']

print(f"\n STATISTICS:")
print(f"  Mean position: {position.mean():.2f}")
print(f"  Median position: {position.median():.2f}")
print(f"  Min (best): {position.min():.2f}")
print(f"  Max (worst): {position.max():.2f}")

print(f"\n PERCENTILES:")
print(f"  10th: {position.quantile(0.10):.2f}")
print(f"  25th: {position.quantile(0.25):.2f}")
print(f"  50th: {position.quantile(0.50):.2f}")
print(f"  75th: {position.quantile(0.75):.2f}")
print(f"  90th: {position.quantile(0.90):.2f}")

print(f"\n BUCKETING (for rule):")
pos_buckets = pd.cut(position, bins=[0, 3, 6, 10, 20, 1000],
                     labels=['top3', 'top6', 'top10', 'top20', 'below20'])
print(pos_buckets.value_counts().sort_index())

print(f"\n  FEATURED SNIPPETS (Position < 1):")
snippets = len(position[position < 1])
print(f"  Articles at position < 1: {snippets} ({snippets/len(position)*100:.1f}%)")
print(f"  These are featured snippets")
print(f"  Rule treats them as position 1, but CTR is actually 0-2%")

print(f"\n INSIGHT:")
print(f"  Most articles rank outside top 20")
print(f"  Featured snippets {snippets/len(position)*100:.1f}% need special handling")
print(f"  Position distribution supports 'harder to improve top 3' logic")

print("\n" + "="*80)

SECTION 1: KEY FIELD DISTRIBUTIONS

SIGNAL 1: IMPRESSIONS (Search Volume)

 STATISTICS:
  Count: 439,193
  Mean: 229.2
  Median: 84.0
  Std Dev: 668.0
  Min: 10
  Max: 245826

 PERCENTILES:
  10th: 18
  25th: 34
  50th (median): 84
  75th: 225
  90th: 537
  99th: 2138

 BUCKETING (for rule):
gsc_impressions
LOW     241714
MED     149429
HIGH     48049
Name: count, dtype: int64

  HEAVY TAIL:
  Top 1% articles: 2138+ impressions
  Count: 4392 articles
  Dominance: 16.7% of total volume

 INSIGHT:
  Volume distribution is RIGHT-SKEWED
  Majority: 100-500 impressions (MED)
  Small tail: 500+ impressions (HIGH) but HIGH volume dominates impact

SIGNAL 2: CTR GAP (Expected - Actual)

 STATISTICS:
  Mean gap: 0.0881 (8.81%)
  Median gap: 0.0759 (7.59%)
  Std Dev: 0.0588
  Min: -0.2929 (over-performing)
  Max: 0.3200 (massive gap)

 PERCENTILES:
  10th: 0.0359
  25th: 0.0500
  50th: 0.0759
  75th: 0.1141
  90th: 0.1608

 BUCKETING (for rule):
ctr_gap
LOW      52859
MED     155186
HIGH    2311

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.